# Ouroboros3D — Pipeline (Dataset Multi-View)

Implementasi paper **"Ouroboros3D: Image-to-3D Generation via 3D-aware Recursive Diffusion"** (CVPR 2025).

Alur: **upload kode → siapkan dataset (pilih sumber) → konversi → training → visualisasi & testing**.
Dua pilihan sumber data (section 2): **A) Google Drive** (foto sendiri) atau **B) SHREC 2026** (multi-view + pose kamera). Keduanya menulis ke `data/real`, sehingga langkah training & visualisasi dipakai bersama.

File `.py` (upload via zip di cell `[0]`):
- `prepare_real_data.py` — konversi foto Drive (pose turntable, mask siluet)
- `prepare_shrec.py` — konversi SHREC (pose COLMAP nyata)
- `dataset.py` / `model.py` / `train.py` — loader, arsitektur, training

## 1. Setup

In [ ]:
# [0] Upload file program (ouroboros3d_project.zip) — jalankan PALING AWAL
# Zip berisi: dataset.py, model.py, train.py, prepare_real_data.py, prepare_shrec.py, requirements.txt
from google.colab import files
import os, zipfile

for f in ['dataset.py', 'model.py', 'train.py',
          'prepare_real_data.py', 'prepare_shrec.py', 'requirements.txt']:
    if os.path.exists(f):
        os.remove(f)

print('Pilih ouroboros3d_project.zip...')
uploaded = files.upload()
zip_name = next(k for k in uploaded if k.endswith('.zip'))
print(f'{zip_name} ({os.path.getsize(zip_name)/1024:.1f} KB)')

with zipfile.ZipFile(zip_name) as z:
    z.extractall('.')
print('Ekstrak Berhasil')
!ls *.py *.txt 2>/dev/null

In [1]:
!nvidia-smi || echo "No GPU — akan jalan di CPU"

Tue May  5 13:00:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.57                 Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   48C    P3             12W /   35W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Dependencies
!pip install torch torchvision numpy matplotlib tqdm -q
!pip install "rembg[cpu]" -q
# rembg sering mengacak versi Pillow di Colab (error "_Ink" / PIL._typing rusak).
# Pasang ulang Pillow KONSISTEN setelah rembg:
!pip install -q --force-reinstall "pillow"

import importlib.util
has_rembg = importlib.util.find_spec("rembg") is not None
has_ort   = importlib.util.find_spec("onnxruntime") is not None
print(f"\nrembg={has_rembg}, onnxruntime={has_ort}")
print("\n" + "=" * 60)
print(">>> WAJIB SEKALI: menu Runtime -> Restart session <<<")
print(">>> lalu jalankan ulang mulai dari cell [0].          <<<")
print(">>> (biar Pillow yang benar termuat; abaikan warning RAPIDS) <<<")
print("=" * 60)

## 2. Sumber Dataset — pilih SALAH SATU

Ada 2 pilihan sumber data. **Keduanya menulis ke `data/real`**, jadi setelah konversi, langkah `[3]`–`[8]` (sanity, training, loss, visualisasi) **identik** untuk keduanya — tidak diduplikat.

- **2A. Google Drive** (foto multi-view sendiri) → cell `[1]`–`[2]` di bawah.
- **2B. SHREC 2026** (multi-view + pose kamera COLMAP, kualitas 3D jauh lebih baik) → cell `[S1]`–`[S2]`.

> Jalankan **salah satu** saja (2A atau 2B), lalu lanjut ke cell `[3]`.

---
### 2A. Google Drive
Folder `archive`: tiap objek berisi foto multi-view, **tanpa** pose kamera (pose diasumsikan turntable → hasil 3D terbatas).
1. `[1]` Mount Drive (*Shared with me* → *Add shortcut to Drive* dulu).
2. `[2]` Konversi (`prepare_real_data.py`): resize, pose turntable, mask siluet (rembg).

In [ ]:
# [1] Ambil dataset dari Google Drive (mount — paling stabil, tanpa limit/permission gdown)
#
# PENTING: foldermu ada di "Dibagikan kepada saya" (Shared with me), yang TIDAK otomatis
# muncul saat mount. Lakukan dulu di Google Drive:
#   Dibagikan kepada saya -> klik kanan folder 'archive' -> "Add shortcut to Drive" -> My Drive
#
from google.colab import drive
drive.mount('/content/drive')

import os
SRC = "/content/drive/MyDrive/archive"   # <-- ganti kalau shortcut-nya kamu taruh di subfolder
assert os.path.isdir(SRC), (
    f"Folder tidak ditemukan: {SRC}\n"
    "Pastikan sudah 'Add shortcut to Drive' untuk folder 'archive' (Shared with me)."
)

print(f"SRC = {SRC}\nObjek terdeteksi:")
for d in sorted(os.listdir(SRC)):
    p = os.path.join(SRC, d)
    if os.path.isdir(p):
        n_img = len([f for f in os.listdir(p)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  {d}/  ({n_img} gambar)")

# Alternatif gdown (kalau tak mau mount) — perlu SEMUA file 'Anyone with the link':
# !gdown --folder "https://drive.google.com/drive/folders/1NHl9gv7x9eDbLeL2G8Wpsr6oHNtavopK" -O data/drive_download --remaining-ok
# SRC = "data/drive_download"

In [ ]:
# [2] Konversi ke format loader (camera_poses + CCM placeholder + mask siluet)
# Mask pakai rembg/U2Net (default) -> segmentasi objek akurat utk foto background alami.
#   (pertama kali: unduh model U2Net ~170MB. Kalau mau cepat tanpa rembg: --mask-method border)
# --exclude images : lewati folder 'images'   --subsample even : view tersebar merata
# --num-views 12   : cukup utk model & lebih cepat (naikkan ke 16/24 kalau mau lebih rapat)
!python prepare_real_data.py --input "{SRC}" --output data/real \
    --exclude images --subsample even --num-views 12 --mask-method rembg

### 2B. SHREC 2026 (alternatif — pose kamera nyata)

938 objek heritage, 90 view/objek, **pose kamera COLMAP**. Pose nyata → generator bisa belajar (hasil tak blur seperti data Drive).
Jalankan `[S1]` (download) lalu `[S2]` (konversi). **Lewati 2A** kalau pakai ini.

In [ ]:
# [S1] Download SHREC 2026 dari Kaggle
# Ambil token: kaggle.com -> Settings -> API -> Create New Token (unduh kaggle.json)
from google.colab import files
print("Pilih kaggle.json...")
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle

!kaggle datasets download -d cristianllull/shrec-2026-retrieval-of-high-frequency-geometry \
    -p data/shrec --unzip

# PENTING: lihat struktur folder hasil download dulu, lalu sesuaikan --input di cell [S2]
import os
print("\nIsi data/shrec:")
for d in sorted(os.listdir("data/shrec"))[:15]:
    print("  ", d)

In [ ]:
# [S2] Konversi SHREC -> data/real (pose COLMAP -> camera_poses, + mask rembg)
# Sesuaikan --input ke folder yang berisi subfolder objek (lihat output [S1]).
# --max-objects 60 : caplok 60 objek saja   --num-views 16 : subsample dari 90
!python prepare_shrec.py --input data/shrec --output data/real \
    --num-views 16 --max-objects 60 --mask-method rembg

# ==> Setelah ini, LANJUT ke cell [3]. Langkah [3]-[8] (sanity, training, loss,
#     visualisasi) SAMA dengan section 2A — tidak diduplikat (sama-sama baca data/real).
#     Catatan: pakai num-views 16 di training (cell [3] auto-deteksi).

In [ ]:
# [3] Sanity check loader + preview dataset real
import glob
from PIL import Image
import matplotlib.pyplot as plt
from dataset import build_dataloaders

# deteksi num_views dari objek pertama
n_views = len(glob.glob("data/real/object_000/views/view_*.png"))
print(f"num_views terdeteksi: {n_views}")

tr, va = build_dataloaders("data/real", img_size=64, num_views=n_views,
                           batch_size=2, num_workers=0)
print(f"Train batches: {len(tr)} | Val batches: {len(va)}")

# preview objek pertama
fig, axes = plt.subplots(1, n_views, figsize=(2*n_views, 2.4))
axes = [axes] if n_views == 1 else axes
for i in range(n_views):
    axes[i].imshow(Image.open(f"data/real/object_000/views/view_{i:02d}.png"))
    axes[i].set_title(f"view {i}", fontsize=8); axes[i].axis("off")
plt.suptitle("Dataset Real - object_000", y=1.02); plt.tight_layout(); plt.show()

In [ ]:
# [4] Training pakai dataset real
# num_views otomatis ikut cell [3]. batch-size kecil supaya aman utk dataset kecil.
# --w-ccm 0    : matikan loss CCM (tak ada geometri GT, CCM placeholder nol)
# --w-mask 0.5 : aktifkan loss siluet (alpha vs mask foreground) -> bentuk 3D tersupervisi
!python train.py --data-root data/real --num-views {n_views} --epochs 10 --batch-size 2 \
    --img-size 64 --num-gaussians 128 --use-feedback --joint --w-ccm 0 --w-mask 0.5 --tag real

In [ ]:
# [4b] Loss curve training real
import torch
import matplotlib.pyplot as plt

h = torch.load("checkpoints/ouroboros_real.pt", map_location="cpu", weights_only=False)["history"]
ep = [x["epoch"] for x in h]

plt.figure(figsize=(7, 4))
plt.plot(ep, [x["train"]["total"] for x in h], "b-o", label="train total")
plt.plot(ep, [x["val"]["total"]   for x in h], "r--s", label="val total")
plt.plot(ep, [x["train"]["mv"]    for x in h], "g-^", label="train mv (RGB)")
plt.plot(ep, [x["train"]["mask"]  for x in h], "m-d", label="train mask (siluet)")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Training Curve — Dataset Real"); plt.tight_layout(); plt.show()

### Visualisasi & Testing — Dataset Real

Load checkpoint `ouroboros_real.pt`, jalankan inferensi pada objek validasi dari dataset Drive-mu, lalu tampilkan:
- **RGB**: target GT vs prediksi multi-view vs render 3D Gaussian Splatting
- **Siluet**: alpha (prediksi) vs mask (GT) — bukti geometri tersupervisi
- **3D Gaussians** + metrik kuantitatif (L1 / PSNR, step-1 vs step-2).

In [ ]:
# [5] Load model hasil training real + inferensi
import torch
from dataset import build_dataloaders
from model import Ouroboros3D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_r = torch.load("checkpoints/ouroboros_real.pt", map_location=device, weights_only=False)
cfg = ckpt_r["args"]          # ambil config persis seperti saat training
model_r = Ouroboros3D(
    num_views=cfg["num_views"], base_ch=32, num_gaussians=cfg["num_gaussians"],
    img_size=cfg["img_size"], use_feedback=cfg["use_feedback"],
).to(device)
model_r.load_state_dict(ckpt_r["model_state"])
model_r.eval()

_, val_loader_r = build_dataloaders(
    "data/real", img_size=cfg["img_size"], num_views=cfg["num_views"],
    batch_size=1, num_workers=0,
)
batch_r = next(iter(val_loader_r))
cond_r  = batch_r["cond_image"].to(device)
poses_r = batch_r["poses"].to(device)

with torch.no_grad():
    outs_r = model_r(cond_r, poses_r, num_recursive_steps=2)

print(f"Model real loaded (img_size={cfg['img_size']}, num_views={cfg['num_views']}, "
      f"gaussians={cfg['num_gaussians']}). Inferensi selesai.")

In [ ]:
# [6] Grid prediksi multi-view: RGB + siluet (alpha vs mask)
import numpy as np
import matplotlib.pyplot as plt

def denorm(t):
    return (t.permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1)

o = outs_r[-1]
pred    = o["pred_target"][0].cpu()        # [N-1, 3, H, W]
gt      = batch_r["target_rgb"][0]         # [N-1, 3, H, W]
rend    = o["rendered_rgb"][0].cpu()       # [N, 3, H, W]
alpha   = o["alpha"][0, :, 0].cpu()        # [N, H, W]
gt_mask = batch_r["all_mask"][0, :, 0]     # [N, H, W]

nv = pred.shape[0]   # jumlah target view (N-1)
rows = ["GT target", "Pred (MVGen)", "Rendered 3DGS", "Alpha (pred)", "Mask (GT)"]

fig, axes = plt.subplots(5, nv, figsize=(2.2 * nv, 9), squeeze=False)
for i in range(nv):
    axes[0, i].imshow(denorm(gt[i]))
    axes[1, i].imshow(denorm(pred[i]))
    axes[2, i].imshow(denorm(rend[i + 1]))           # +1: index 0 = conditioning
    axes[3, i].imshow(alpha[i + 1].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[4, i].imshow(gt_mask[i + 1].numpy(), cmap="gray", vmin=0, vmax=1)
    for r in range(5):
        axes[r, i].set_xticks([]); axes[r, i].set_yticks([])
for r, label in enumerate(rows):
    axes[r, 0].set_ylabel(label, fontsize=9)

plt.suptitle("Dataset Real — Output Pipeline (target views)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# [7] Visualisasi 3D Gaussians hasil rekonstruksi objek real
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

g = outs_r[-1]["gaussians"]
pos   = g["pos"][0].cpu().numpy()
rgb   = g["rgb"][0].cpu().numpy()
opa   = g["opacity"][0, :, 0].cpu().numpy()
scale = g["scale"][0, :, 0].cpu().numpy()
keep  = opa > 0.1

fig = plt.figure(figsize=(14, 4.5))
ax1 = fig.add_subplot(131, projection="3d")
ax1.scatter(pos[keep, 0], pos[keep, 1], pos[keep, 2],
            c=rgb[keep].clip(0, 1), s=scale[keep] * 200, alpha=0.6)
ax1.set_title(f"Gaussian 3D ({keep.sum()}/{len(keep)} aktif)")
ax1.set_xlabel("X"); ax1.set_ylabel("Y"); ax1.set_zlabel("Z")

ax2 = fig.add_subplot(132)
ax2.hist(opa, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
ax2.axvline(0.1, color="red", linestyle="--", label="threshold")
ax2.set_title("Distribusi Opacity"); ax2.legend(); ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(133)
ax3.hist(scale, bins=30, color="salmon", edgecolor="white", alpha=0.8)
ax3.set_title("Distribusi Scale"); ax3.grid(True, alpha=0.3)

plt.suptitle("Dataset Real — 3D Gaussian Splatting", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# [8] Testing kuantitatif pada seluruh objek validasi (L1, PSNR, IoU siluet, step-1 vs step-2)
import numpy as np
import torch

def psnr_from_mse(mse):
    return 10.0 * np.log10(1.0 / (mse + 1e-8))

l1_s1, l1_s2, psnr_s1, iou = [], [], [], []
model_r.eval()
with torch.no_grad():
    for b in val_loader_r:
        c = b["cond_image"].to(device)
        p = b["poses"].to(device)
        gt   = b["target_rgb"]                 # [1, N-1, 3, H, W]
        gtm  = b["all_mask"].to(device)        # [1, N, 1, H, W]

        outs = model_r(c, p, num_recursive_steps=2)
        pr1 = outs[0]["pred_target"].cpu()
        pr2 = outs[-1]["pred_target"].cpu()

        l1_s1.append((pr1 - gt).abs().mean().item())
        l1_s2.append((pr2 - gt).abs().mean().item())
        mse = ((pr1 * 0.5 + 0.5) - (gt * 0.5 + 0.5)).pow(2).mean().item()
        psnr_s1.append(psnr_from_mse(mse))

        # IoU siluet: alpha (pred) vs mask (GT)
        a = (outs[-1]["alpha"] > 0.5).float()
        m = (gtm > 0.5).float()
        inter = (a * m).sum().item()
        union = ((a + m) > 0).float().sum().item()
        iou.append(inter / (union + 1e-8))

print("=" * 50)
print(f"  Objek validasi   : {len(l1_s1)}")
print(f"  L1  (step-1)     : {np.mean(l1_s1):.4f}")
print(f"  L1  (step-2)     : {np.mean(l1_s2):.4f}")
print(f"  PSNR (step-1)    : {np.mean(psnr_s1):.2f} dB")
print(f"  IoU siluet       : {np.mean(iou):.3f}")
print("=" * 50)
if np.mean(l1_s2) < np.mean(l1_s1):
    imp = (np.mean(l1_s1) - np.mean(l1_s2)) / np.mean(l1_s1) * 100
    print(f"  Recursive feedback membantu: step-2 lebih baik {imp:.1f}%")
else:
    print("  Step-1 & step-2 setara (model belum converged penuh / epoch sedikit).")